# 05. Preparación del conjunto de datos

Se excluyen únicamente los registros cuyo `texto_procesado` está vacío y se reserva una muestra fija de 100 publicaciones para evaluación independiente.

El dataset principal termina con **exactamente cuatro columnas**: `id`, `fuente`, `texto_real`, `texto_procesado`. La muestra de evaluación es un instrumento separado y añade solamente `etiqueta_polaridad` y `observacion` porque deben ser completadas por una persona.


## Criterio de clasificación de polaridad

Esto **no es una prueba psicológica ni un diagnóstico de la emoción del autor**. La base conceptual es la **valencia afectiva** —dirección agradable/favorable o desagradable/desfavorable— del modelo circumplejo de Russell (1980). La decisión lingüística sigue la guía práctica de anotación de sentimiento de Mohammad (2016).

Para cada publicación se responden estas cinco preguntas:

1. **¿El texto se entiende con la información disponible?** Si no, se deja sin etiqueta y se explica en `observacion`.
2. **¿Expresa una valoración o postura?** Si solo informa, pregunta o describe sin valoración clara, es `neutral`.
3. **¿Sobre qué persona, objeto, hecho o situación recae la valoración?** Esto evita atribuir al autor opiniones que solo está citando.
4. **Considerando el mensaje completo, la valoración predominante es favorable, desfavorable o ninguna?** Favorable = `positiva`; desfavorable = `negativa`; ninguna clara = `neutral`.
5. **¿Negación, ironía, jerga, emojis, citas o partes mixtas cambian la lectura?** Si impiden una decisión responsable, se deja en blanco y se registra la causa.

Referencias: [Russell (1980), *A circumplex model of affect*](https://doi.org/10.1037/h0077714) y [Mohammad (2016), *A Practical Guide to Sentiment Annotation*](https://aclanthology.org/W16-0429/).


## A. Cargar el texto preprocesado

- **Qué hace:** abre la salida del cuaderno 04.
- **Para qué:** preparar solo textos disponibles para clasificación.
- **Por qué:** un texto vacío no contiene evidencia para asignar polaridad.


In [ ]:
from pathlib import Path

if Path("/content").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
    RAIZ = Path("/content/drive/MyDrive/TESIS_DATA_SCIENCE/COLAB")
else:
    RAIZ = Path.cwd()

if not (RAIZ / "TABLAS").is_dir():
    raise FileNotFoundError("Abre este notebook desde la carpeta COLAB o ajusta RAIZ.")
print("Carpeta de trabajo:", RAIZ)

import hashlib
import json
import re
import pandas as pd
from IPython.display import display
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter

ENTRADA = RAIZ / "TABLAS/04_preprocesamiento/datos_preprocesados_v4.xlsx"
datos = pd.read_excel(ENTRADA, sheet_name="datos", keep_default_na=False)
esperadas = ["id", "fuente", "texto_real", "texto_procesado"]
if list(datos.columns) != esperadas:
    raise ValueError("Ejecuta primero el cuaderno 04 actualizado.")

vacios = datos["texto_procesado"].astype(str).str.strip().eq("")
preparado = datos.loc[~vacios, esperadas].copy()
print("Filas recibidas:", len(datos), "| vacías excluidas:", int(vacios.sum()), "| disponibles:", len(preparado))
display(preparado.head(10))


## B. Reservar 100 publicaciones

- **Qué hace:** separa los mismos 100 `id` definidos antes del modelado.
- **Para qué:** evaluar al final con textos bolivianos etiquetados sin mirar predicciones.
- **Por qué:** si estos casos influyen en el entrenamiento o en los ajustes, la evaluación deja de ser independiente.

La clave para detectar textos equivalentes se calcula solo en memoria. No se agrega como columna al dataset.

El tamaño 100 es una decisión operativa para disponer de una prueba independiente manejable; **no demuestra representatividad estadística de toda la población boliviana**. La tesis debe informarlo como limitación y reportar el número realmente etiquetado por clase.


In [ ]:
IDS_PRUEBA = [2596, 624, 930, 1290, 2242, 783, 2250, 261, 1458, 2561, 2464, 1549, 2241, 481, 1813, 305, 2484, 1630, 385, 2572, 1092, 1286, 2522, 2481, 913, 1163, 1880, 1333, 597, 1035, 1268, 1639, 618, 2140, 1730, 933, 656, 80, 1921, 2025, 1615, 1522, 1981, 935, 2054, 1113, 2495, 524, 1789, 1227, 1626, 2108, 169, 1839, 24, 953, 1879, 1005, 2247, 2275, 2411, 2125, 1741, 1146, 2549, 2537, 62, 2606, 245, 353, 103, 2049, 834, 215, 1300, 525, 1536, 2016, 299, 1538, 354, 566, 381, 2307, 2254, 2217, 2349, 666, 322, 1011, 352, 66, 486, 1097, 487, 485, 2599, 125, 18, 1254]
if len(IDS_PRUEBA) != 100 or len(set(IDS_PRUEBA)) != 100:
    raise ValueError("La lista de prueba contiene identificadores repetidos.")
if not set(IDS_PRUEBA).issubset(set(preparado["id"])):
    raise ValueError("Algún id reservado ya no existe; revisa las etapas anteriores.")

def clave_temporal(texto):
    canon = re.sub(r"\s+", " ", str(texto)).strip().casefold()
    return hashlib.sha256(canon.encode("utf-8")).hexdigest()

claves = preparado["texto_procesado"].map(clave_temporal)
muestra = preparado.loc[preparado["id"].isin(IDS_PRUEBA)].copy()
muestra["etiqueta_polaridad"] = ""
muestra["observacion"] = ""

claves_prueba = set(claves[preparado["id"].isin(IDS_PRUEBA)])
claves_resto = set(claves[~preparado["id"].isin(IDS_PRUEBA)])
if claves_prueba & claves_resto:
    raise ValueError("Un texto equivalente aparece en prueba y en el resto del corpus.")
if len(muestra) != 100:
    raise ValueError("La muestra debe contener 100 publicaciones.")

print("Muestra:", len(muestra), "| distribución:", muestra["fuente"].value_counts().to_dict())
display(muestra.head(10))


## C. Guardar el dataset y la hoja de anotación

- **Qué hace:** guarda el dataset de cuatro columnas y crea la muestra con listas desplegables.
- **Para qué:** alimentar por separado la inferencia automática y la evaluación humana.
- **Por qué:** separar ambos archivos impide confundir sugerencias del modelo con etiquetas humanas de referencia.


In [ ]:
CARPETA = RAIZ / "TABLAS/05_preparacion"
CARPETA.mkdir(parents=True, exist_ok=True)
DATASET = CARPETA / "dataset_preparado_v4.xlsx"
PRUEBA = CARPETA / "muestra_prueba_100_v4.xlsx"
RESUMEN = CARPETA / "resumen_preparacion_v4.json"

preparado.to_excel(DATASET, sheet_name="datos", index=False)

if PRUEBA.exists():
    anterior = pd.read_excel(PRUEBA, sheet_name="evaluacion", keep_default_na=False)
    if set(anterior["id"]) != set(muestra["id"]):
        raise ValueError("La muestra existente contiene otros identificadores.")
    anotaciones = anterior.set_index("id")[["etiqueta_polaridad", "observacion"]]
    muestra = muestra.set_index("id")
    muestra[["etiqueta_polaridad", "observacion"]] = anotaciones
    muestra = muestra.reset_index()

with pd.ExcelWriter(PRUEBA, engine="openpyxl") as escritor:
    muestra.to_excel(escritor, sheet_name="evaluacion", index=False)
    hoja = escritor.sheets["evaluacion"]
    hoja.freeze_panes = "A2"
    hoja.auto_filter.ref = hoja.dimensions
    hoja.column_dimensions["C"].width = 70
    hoja.column_dimensions["D"].width = 70
    hoja.column_dimensions["F"].width = 45
    validacion = DataValidation(type="list", formula1='"positiva,negativa,neutral"', allow_blank=True)
    hoja.add_data_validation(validacion)
    col = get_column_letter(muestra.columns.get_loc("etiqueta_polaridad") + 1)
    validacion.add(f"{col}2:{col}{hoja.max_row}")

resumen = {
    "version": "v4",
    "filas_recibidas": int(len(datos)),
    "textos_vacios_excluidos": int(vacios.sum()),
    "filas_dataset": int(len(preparado)),
    "filas_prueba": int(len(muestra)),
    "prueba_por_fuente": {str(k): int(v) for k, v in muestra["fuente"].value_counts().items()},
    "columnas_dataset": list(preparado.columns),
    "regla": "La muestra de prueba no se usa para entrenar ni ajustar decisiones.",
}
RESUMEN.write_text(json.dumps(resumen, ensure_ascii=False, indent=2), encoding="utf-8")
print("Dataset:", DATASET)
print("Muestra que debes etiquetar:", PRUEBA)
print("Columnas del dataset:", list(preparado.columns))
